1. Load experimental model
2. Reuse/import simulation functions where possible
3. Run 100 simulations first
4. Run 1,000 simulations
5. Compare confederation winner probability
6. Save experiment CSV

# Experiment A2 — Test no-conf clean model in tournament simulation

In this experiment, we test the model trained without confederation features.

Goal:

- Load the `no_conf_clean` Poisson models
- Use the same tournament simulation logic
- Run Monte Carlo simulations
- Check whether removing confederation features reduces the CONMEBOL bias

In [4]:
from pathlib import Path
import pickle
import time

import numpy as np
import pandas as pd
from scipy.stats import poisson

In [5]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REFERENCE_DIR = DATA_DIR / "reference"
RAW_DIR = DATA_DIR / "raw"

EXPERIMENT_MODELS_DIR = PROJECT_ROOT / "models" / "experiments" / "no_conf_clean"
EXPERIMENT_OUTPUT_DIR = PROCESSED_DIR / "experiments"
EXPERIMENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Experiment model dir:", EXPERIMENT_MODELS_DIR)
print("Experiment output dir:", EXPERIMENT_OUTPUT_DIR)

Project root: /Users/riadanas/Desktop/Football WC2026
Experiment model dir: /Users/riadanas/Desktop/Football WC2026/models/experiments/no_conf_clean
Experiment output dir: /Users/riadanas/Desktop/Football WC2026/data/processed/experiments


In [6]:
##### Load experiment model

with open(EXPERIMENT_MODELS_DIR / "poisson_home.pkl", "rb") as f:
    model_home = pickle.load(f)

with open(EXPERIMENT_MODELS_DIR / "poisson_away.pkl", "rb") as f:
    model_away = pickle.load(f)

with open(EXPERIMENT_MODELS_DIR / "feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

print("Loaded no-conf clean models")
print("Feature columns:", feature_columns)

Loaded no-conf clean models
Feature columns: ['const', 'home_elo_pre', 'away_elo_pre', 'tournament_weight', 'neutral']


In [7]:
#### Load data

df_fixtures = pd.read_csv(INTERIM_DIR / "wc2026_fixtures.csv")
df_match_features = pd.read_csv(PROCESSED_DIR / "df_match_features.csv")
df_groups = pd.read_csv(REFERENCE_DIR / "group_stages.csv", sep=";")
df_knockout = pd.read_csv(REFERENCE_DIR / "fixtures_knockout_wc2026.csv")
df_confederations = pd.read_csv(REFERENCE_DIR / "FIFA_confederations.csv")
df_form = pd.read_csv(PROCESSED_DIR / "df_form_2026.csv")
df_fifa_rank = pd.read_csv(RAW_DIR / "wc_2026_48_teams_fifa_rank_change_corrected.csv")

print("Fixtures:", df_fixtures.shape)
print("Match features:", df_match_features.shape)
print("Groups:", df_groups.shape)
print("Knockout:", df_knockout.shape)

Fixtures: (72, 9)
Match features: (49048, 16)
Groups: (48, 3)
Knockout: (32, 8)


In [8]:
#### Build lookups

df_match_features["date"] = pd.to_datetime(df_match_features["date"])

home_elo = df_match_features[["date", "home_team", "home_elo_pre"]].rename(
    columns={
        "home_team": "team",
        "home_elo_pre": "elo",
    }
)

away_elo = df_match_features[["date", "away_team", "away_elo_pre"]].rename(
    columns={
        "away_team": "team",
        "away_elo_pre": "elo",
    }
)

df_team_elo = pd.concat([home_elo, away_elo], ignore_index=True)

df_latest_elo = (
    df_team_elo
    .sort_values("date")
    .drop_duplicates(subset="team", keep="last")
    .reset_index(drop=True)
)

team_to_elo = dict(zip(df_latest_elo["team"], df_latest_elo["elo"]))

team_to_group = dict(zip(df_groups["nation"], df_groups["group"]))

team_to_confederation = dict(
    zip(df_confederations["nation"], df_confederations["confederation"])
)

team_to_form = dict(zip(df_form["team"], df_form["form_score"]))

team_to_fifa_rank = dict(
    zip(df_fifa_rank["Nation"], df_fifa_rank["FIFA_2026_rank"])
)

team_to_fifa_rank_change = dict(
    zip(df_fifa_rank["Nation"], df_fifa_rank["rank_change"])
)

print("Elo teams:", len(team_to_elo))
print("Group teams:", len(team_to_group))
print("Confederation teams:", len(team_to_confederation))

Elo teams: 322
Group teams: 48
Confederation teams: 48


In [9]:
#### Add group info to fixtures

df_group_fixtures = df_fixtures.copy()
df_group_fixtures["group"] = df_group_fixtures["home_team"].map(team_to_group)

missing_group_fixtures = df_group_fixtures[df_group_fixtures["group"].isna()]

print("Fixtures with missing group:", len(missing_group_fixtures))
display(missing_group_fixtures)

print(df_group_fixtures["group"].value_counts().sort_index())

Fixtures with missing group: 0


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,group


group
A    6
B    6
C    6
D    6
E    6
F    6
G    6
H    6
I    6
J    6
K    6
L    6
Name: count, dtype: int64


### Build prediction logic for no-conf clean model

In [10]:
#### Build match features

def build_match_features_no_conf_clean(home_team, away_team, neutral=True, tournament_weight=5):
    """
    Build one prediction row for the no-conf clean model.
    """

    home_elo = team_to_elo.get(home_team)
    away_elo = team_to_elo.get(away_team)

    if home_elo is None:
        raise ValueError(f"Missing Elo for home team: {home_team}")

    if away_elo is None:
        raise ValueError(f"Missing Elo for away team: {away_team}")

    row = pd.DataFrame([{
        "home_elo_pre": home_elo,
        "away_elo_pre": away_elo,
        "tournament_weight": tournament_weight,
        "neutral": int(neutral),
    }])

    row["const"] = 1.0

    X = row.reindex(columns=feature_columns, fill_value=0)
    X = X.astype(float)

    return X

In [12]:
#### Predict one match

def predict_match_no_conf_clean(
    home_team,
    away_team,
    neutral=True,
    tournament_weight=5,
    max_goals=10,
):
    """
    Predict expected goals and W/D/L probabilities using no-conf clean model.
    """

    X = build_match_features_no_conf_clean(
        home_team=home_team,
        away_team=away_team,
        neutral=neutral,
        tournament_weight=tournament_weight,
    )

    home_xg = float(model_home.predict(X)[0])
    away_xg = float(model_away.predict(X)[0])

    score_probs = []

    home_win_prob = 0.0
    draw_prob = 0.0
    away_win_prob = 0.0

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):
            prob = float(
                poisson.pmf(home_goals, home_xg)
                * poisson.pmf(away_goals, away_xg)
            )

            score_probs.append({
                "home_goals": home_goals,
                "away_goals": away_goals,
                "probability": prob,
            })

            if home_goals > away_goals:
                home_win_prob += prob
            elif home_goals == away_goals:
                draw_prob += prob
            else:
                away_win_prob += prob

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "home_win_prob": home_win_prob,
        "draw_prob": draw_prob,
        "away_win_prob": away_win_prob,
        "score_probs": pd.DataFrame(score_probs),
    }

In [13]:
#### Quick prediction sanity check

test_matches = [
    ("Argentina", "Brazil"),
    ("France", "Spain"),
    ("Morocco", "Portugal"),
    ("England", "Iran"),
    ("Mexico", "South Africa"),
]

for home_team, away_team in test_matches:
    pred = predict_match_no_conf_clean(home_team, away_team)

    print(f"\n{home_team} vs {away_team}")
    print(f"Home xG: {pred['home_xg']:.2f}")
    print(f"Away xG: {pred['away_xg']:.2f}")
    print(f"Home win: {pred['home_win_prob']:.1%}")
    print(f"Draw: {pred['draw_prob']:.1%}")
    print(f"Away win: {pred['away_win_prob']:.1%}")


Argentina vs Brazil
Home xG: 1.38
Away xG: 0.78
Home win: 51.0%
Draw: 27.7%
Away win: 21.3%

France vs Spain
Home xG: 0.89
Away xG: 1.21
Home win: 27.1%
Draw: 29.5%
Away win: 43.4%

Morocco vs Portugal
Home xG: 1.06
Away xG: 1.13
Home win: 33.5%
Draw: 29.2%
Away win: 37.3%

England vs Iran
Home xG: 1.72
Away xG: 0.68
Home win: 62.5%
Draw: 23.2%
Away win: 14.3%

Mexico vs South Africa
Home xG: 2.11
Away xG: 0.62
Home win: 72.2%
Draw: 18.3%
Away win: 9.6%


### Group Stage Logic

In [15]:
#### Simulate one match

def simulate_match_no_conf_clean(home_team, away_team, neutral=True, tournament_weight=5):
    """
    Simulate one group-stage match.
    Draws are allowed.
    """

    pred = predict_match_no_conf_clean(
        home_team=home_team,
        away_team=away_team,
        neutral=neutral,
        tournament_weight=tournament_weight,
    )

    home_goals = int(np.random.poisson(pred["home_xg"]))
    away_goals = int(np.random.poisson(pred["away_xg"]))

    if home_goals > away_goals:
        result = "H"
        winner = home_team
    elif away_goals > home_goals:
        result = "A"
        winner = away_team
    else:
        result = "D"
        winner = None

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_xg": pred["home_xg"],
        "away_xg": pred["away_xg"],
        "home_goals": home_goals,
        "away_goals": away_goals,
        "result": result,
        "winner": winner,
    }

In [16]:
#### Group table helpers

def create_empty_group_table(group_teams):
    return pd.DataFrame({
        "team": group_teams,
        "played": 0,
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "goals_for": 0,
        "goals_against": 0,
        "goal_difference": 0,
        "points": 0,
    })


def update_group_table(table, match):
    table = table.copy()

    home_team = match["home_team"]
    away_team = match["away_team"]
    home_goals = match["home_goals"]
    away_goals = match["away_goals"]

    table.loc[table["team"] == home_team, "played"] += 1
    table.loc[table["team"] == away_team, "played"] += 1

    table.loc[table["team"] == home_team, "goals_for"] += home_goals
    table.loc[table["team"] == home_team, "goals_against"] += away_goals

    table.loc[table["team"] == away_team, "goals_for"] += away_goals
    table.loc[table["team"] == away_team, "goals_against"] += home_goals

    if home_goals > away_goals:
        table.loc[table["team"] == home_team, "wins"] += 1
        table.loc[table["team"] == away_team, "losses"] += 1
        table.loc[table["team"] == home_team, "points"] += 3

    elif away_goals > home_goals:
        table.loc[table["team"] == away_team, "wins"] += 1
        table.loc[table["team"] == home_team, "losses"] += 1
        table.loc[table["team"] == away_team, "points"] += 3

    else:
        table.loc[table["team"] == home_team, "draws"] += 1
        table.loc[table["team"] == away_team, "draws"] += 1
        table.loc[table["team"] == home_team, "points"] += 1
        table.loc[table["team"] == away_team, "points"] += 1

    table["goal_difference"] = table["goals_for"] - table["goals_against"]

    return table


def rank_group_table(table):
    table = table.copy()

    table["random_tiebreaker"] = np.random.random(len(table))

    table = (
        table
        .sort_values(
            by=["points", "goal_difference", "goals_for", "random_tiebreaker"],
            ascending=[False, False, False, False],
        )
        .reset_index(drop=True)
    )

    table["group_rank"] = table.index + 1

    return table.drop(columns=["random_tiebreaker"])

In [17]:
#### Simulate one group and full group stage

def simulate_group_no_conf_clean(group_name):
    group_teams = (
        df_groups[df_groups["group"] == group_name]
        .sort_values("position")["nation"]
        .tolist()
    )

    group_matches = df_group_fixtures[df_group_fixtures["group"] == group_name]

    table = create_empty_group_table(group_teams)
    simulated_matches = []

    for _, row in group_matches.iterrows():
        match = simulate_match_no_conf_clean(
            home_team=row["home_team"],
            away_team=row["away_team"],
            neutral=bool(row["neutral"]),
        )

        match["group"] = group_name
        simulated_matches.append(match)

        table = update_group_table(table, match)

    ranked_table = rank_group_table(table)
    ranked_table["group"] = group_name

    return ranked_table, pd.DataFrame(simulated_matches)


def simulate_group_stage_no_conf_clean():
    all_group_tables = []
    all_group_matches = []

    for group_name in sorted(df_groups["group"].unique()):
        group_table, group_matches = simulate_group_no_conf_clean(group_name)

        all_group_tables.append(group_table)
        all_group_matches.append(group_matches)

    df_group_tables = pd.concat(all_group_tables, ignore_index=True)
    df_group_matches = pd.concat(all_group_matches, ignore_index=True)

    return df_group_tables, df_group_matches

In [18]:
#### Test one group

group_table_test, group_matches_test = simulate_group_no_conf_clean("C")

display(group_matches_test)
display(group_table_test)

,home_team,away_team,home_xg,away_xg,home_goals,away_goals,result,winner,group
0,Brazil,Morocco,1.318353,0.887141,2,0,H,Brazil,C
1,Haiti,Scotland,1.009466,1.414121,3,4,A,Scotland,C
2,Scotland,Morocco,0.940595,1.379729,1,1,D,NaN,C
3,Brazil,Haiti,2.305654,0.537903,3,0,H,Brazil,C
4,Scotland,Brazil,0.806460,1.583441,1,3,A,Brazil,C
5,Morocco,Haiti,2.047398,0.628333,0,2,A,Haiti,C


,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,Brazil,3,3,0,0,8,1,7,9,1,C
1,Scotland,3,1,1,1,6,7,-1,4,2,C
2,Haiti,3,1,0,2,5,7,-2,3,3,C
3,Morocco,3,0,1,2,1,5,-4,1,4,C


### Qualification and bracket logic

In [19]:
#### Qualifiers 

def get_direct_qualifiers(df_group_tables):
    direct_qualifiers = {}

    top_two = df_group_tables[df_group_tables["group_rank"].isin([1, 2])].copy()

    for _, row in top_two.iterrows():
        slot = f"{int(row['group_rank'])}{row['group']}"
        direct_qualifiers[slot] = row["team"]

    return direct_qualifiers


def get_best_third_placed_teams(df_group_tables, n_teams=8):
    third_placed = df_group_tables[df_group_tables["group_rank"] == 3].copy()

    third_placed["fifa_rank"] = third_placed["team"].map(team_to_fifa_rank)

    best_third_placed = (
        third_placed
        .sort_values(
            by=["points", "goal_difference", "goals_for", "fifa_rank"],
            ascending=[False, False, False, True],
        )
        .head(n_teams)
        .reset_index(drop=True)
    )

    best_third_placed["third_place_rank"] = best_third_placed.index + 1

    return best_third_placed


def get_qualified_teams(df_group_tables):
    direct_qualifiers = get_direct_qualifiers(df_group_tables)
    best_third_placed = get_best_third_placed_teams(df_group_tables)

    direct_teams = list(direct_qualifiers.values())
    third_placed_teams = best_third_placed["team"].tolist()

    qualified_teams = direct_teams + third_placed_teams

    return direct_qualifiers, best_third_placed, qualified_teams

In [20]:
#### Third-place assignment and R32 bracket

from itertools import permutations


def assign_third_place_slots(third_place_slots, best_third_placed):
    third_teams = best_third_placed[["team", "group", "third_place_rank"]].copy()

    if len(third_place_slots) != len(third_teams):
        raise ValueError(
            f"Third-place slots ({len(third_place_slots)}) do not match "
            f"third-place teams ({len(third_teams)})."
        )

    for perm in permutations(third_teams.to_dict("records")):
        assignment = {}
        valid = True

        for slot, team_row in zip(third_place_slots, perm):
            allowed_groups = list(str(slot).replace("3", ""))

            if team_row["group"] not in allowed_groups:
                valid = False
                break

            assignment[slot] = team_row["team"]

        if valid:
            return assignment

    raise ValueError("No valid third-place assignment found.")


def build_round_32_bracket(df_round_32, direct_qualifiers, best_third_placed):
    df_round_32 = df_round_32.copy()

    third_place_slots = []

    for col in ["home_slot", "away_slot"]:
        slots = (
            df_round_32[col]
            .astype(str)
            .loc[lambda s: s.str.startswith("3")]
            .unique()
            .tolist()
        )
        third_place_slots.extend(slots)

    third_place_slots = list(dict.fromkeys(third_place_slots))

    third_place_assignment = assign_third_place_slots(
        third_place_slots=third_place_slots,
        best_third_placed=best_third_placed,
    )

    filled_matches = []

    for _, row in df_round_32.iterrows():
        home_slot = str(row["home_slot"])
        away_slot = str(row["away_slot"])

        if home_slot in direct_qualifiers:
            home_team = direct_qualifiers[home_slot]
        elif home_slot.startswith("3"):
            home_team = third_place_assignment[home_slot]
        else:
            raise ValueError(f"Unsupported home slot: {home_slot}")

        if away_slot in direct_qualifiers:
            away_team = direct_qualifiers[away_slot]
        elif away_slot.startswith("3"):
            away_team = third_place_assignment[away_slot]
        else:
            raise ValueError(f"Unsupported away slot: {away_slot}")

        match = row.to_dict()
        match["home_team"] = home_team
        match["away_team"] = away_team

        filled_matches.append(match)

    return pd.DataFrame(filled_matches)

### Knockout logic

In [21]:
def simulate_knockout_match_no_conf_clean(home_team, away_team, neutral=True, tournament_weight=5):
    pred = predict_match_no_conf_clean(
        home_team=home_team,
        away_team=away_team,
        neutral=neutral,
        tournament_weight=tournament_weight,
    )

    home_goals = int(np.random.poisson(pred["home_xg"]))
    away_goals = int(np.random.poisson(pred["away_xg"]))

    if home_goals > away_goals:
        winner = home_team
        loser = away_team
        result_type = "normal_time"

    elif away_goals > home_goals:
        winner = away_team
        loser = home_team
        result_type = "normal_time"

    else:
        home_strength = pred["home_win_prob"]
        away_strength = pred["away_win_prob"]

        home_advance_prob = home_strength / (home_strength + away_strength)

        if np.random.random() < home_advance_prob:
            winner = home_team
            loser = away_team
        else:
            winner = away_team
            loser = home_team

        result_type = "draw_resolved"

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_xg": pred["home_xg"],
        "away_xg": pred["away_xg"],
        "home_goals": home_goals,
        "away_goals": away_goals,
        "winner": winner,
        "loser": loser,
        "result_type": result_type,
        "home_win_prob": pred["home_win_prob"],
        "draw_prob": pred["draw_prob"],
        "away_win_prob": pred["away_win_prob"],
    }


def simulate_knockout_round_no_conf_clean(df_round_matches):
    simulated_results = []

    for _, row in df_round_matches.iterrows():
        match_result = simulate_knockout_match_no_conf_clean(
            home_team=row["home_team"],
            away_team=row["away_team"],
            neutral=True,
        )

        match_result["match_id"] = row["match_id"]
        match_result["round"] = row["round"]
        match_result["winner_advances_to"] = row["winner_advances_to"]
        match_result["loser_advances_to"] = row["loser_advances_to"]

        simulated_results.append(match_result)

    return pd.DataFrame(simulated_results)


def build_next_round(df_knockout, previous_round_results, next_round_name):
    df_next_round = df_knockout[df_knockout["round"] == next_round_name].copy()

    next_match_teams = {}

    for _, row in previous_round_results.iterrows():
        next_match_id = row["winner_advances_to"]
        winner = row["winner"]

        if pd.isna(next_match_id):
            continue

        if next_match_id not in next_match_teams:
            next_match_teams[next_match_id] = []

        next_match_teams[next_match_id].append(winner)

    filled_matches = []

    for _, row in df_next_round.iterrows():
        match_id = row["match_id"]
        teams = next_match_teams.get(match_id, [])

        if len(teams) != 2:
            raise ValueError(
                f"Expected 2 teams for match {match_id}, got {len(teams)}: {teams}"
            )

        match = row.to_dict()
        match["home_team"] = teams[0]
        match["away_team"] = teams[1]

        filled_matches.append(match)

    return pd.DataFrame(filled_matches)


def simulate_knockout_stage_no_conf_clean(df_round_32_filled):
    df_r32_results = simulate_knockout_round_no_conf_clean(df_round_32_filled)

    df_r16_filled = build_next_round(df_knockout, df_r32_results, "R16")
    df_r16_results = simulate_knockout_round_no_conf_clean(df_r16_filled)

    df_qf_filled = build_next_round(df_knockout, df_r16_results, "QF")
    df_qf_results = simulate_knockout_round_no_conf_clean(df_qf_filled)

    df_sf_filled = build_next_round(df_knockout, df_qf_results, "SF")
    df_sf_results = simulate_knockout_round_no_conf_clean(df_sf_filled)

    df_final_filled = build_next_round(df_knockout, df_sf_results, "Final")
    df_final_results = simulate_knockout_round_no_conf_clean(df_final_filled)

    df_knockout_results = pd.concat(
        [
            df_r32_results,
            df_r16_results,
            df_qf_results,
            df_sf_results,
            df_final_results,
        ],
        ignore_index=True,
    )

    winner = df_final_results["winner"].iloc[0]
    runner_up = df_final_results["loser"].iloc[0]

    return df_knockout_results, winner, runner_up

In [22]:
#### Full tournament simulation

def simulate_tournament_no_conf_clean():
    df_group_tables, df_group_matches = simulate_group_stage_no_conf_clean()

    direct_qualifiers, best_third_placed, qualified_teams = get_qualified_teams(
        df_group_tables
    )

    df_round_32 = df_knockout[df_knockout["round"] == "R32"].copy()

    df_round_32_filled = build_round_32_bracket(
        df_round_32=df_round_32,
        direct_qualifiers=direct_qualifiers,
        best_third_placed=best_third_placed,
    )

    df_knockout_results, winner, runner_up = simulate_knockout_stage_no_conf_clean(
        df_round_32_filled=df_round_32_filled
    )

    r32_teams = qualified_teams

    r16_teams = (
        df_knockout_results
        .loc[df_knockout_results["round"] == "R32", "winner"]
        .tolist()
    )

    qf_teams = (
        df_knockout_results
        .loc[df_knockout_results["round"] == "R16", "winner"]
        .tolist()
    )

    sf_teams = (
        df_knockout_results
        .loc[df_knockout_results["round"] == "QF", "winner"]
        .tolist()
    )

    final_teams = (
        df_knockout_results
        .loc[df_knockout_results["round"] == "SF", "winner"]
        .tolist()
    )

    summary = {
        "winner": winner,
        "runner_up": runner_up,
        "r32_teams": r32_teams,
        "r16_teams": r16_teams,
        "qf_teams": qf_teams,
        "sf_teams": sf_teams,
        "final_teams": final_teams,
    }

    return {
        "summary": summary,
        "group_tables": df_group_tables,
        "group_matches": df_group_matches,
        "round_32_bracket": df_round_32_filled,
        "knockout_results": df_knockout_results,
    }

In [23]:
#### Test one tournament

tournament_test = simulate_tournament_no_conf_clean()
summary_test = tournament_test["summary"]

print("Winner:", summary_test["winner"])
print("Runner-up:", summary_test["runner_up"])
print("R32 teams:", len(summary_test["r32_teams"]))
print("R16 teams:", len(summary_test["r16_teams"]))
print("QF teams:", len(summary_test["qf_teams"]))
print("SF teams:", len(summary_test["sf_teams"]))
print("Final teams:", len(summary_test["final_teams"]))

display(tournament_test["knockout_results"].tail())

Winner: Senegal
Runner-up: Spain
R32 teams: 32
R16 teams: 16
QF teams: 8
SF teams: 4
Final teams: 2


,home_team,away_team,home_xg,away_xg,home_goals,away_goals,winner,loser,result_type,home_win_prob,draw_prob,away_win_prob,match_id,round,winner_advances_to,loser_advances_to
26,Ecuador,Mexico,1.283013,0.929312,2,1,Ecuador,Mexico,normal_time,0.446105,0.284394,0.269501,M99,QF,M102,NaN
27,Spain,Switzerland,1.845011,0.585887,1,1,Spain,Switzerland,draw_resolved,0.677315,0.212036,0.110645,M100,QF,M102,NaN
28,Senegal,Panama,1.393748,0.912874,2,1,Senegal,Panama,normal_time,0.480993,0.272798,0.246208,M101,SF,M104,M103
29,Ecuador,Spain,0.749263,1.503994,0,2,Spain,Ecuador,normal_time,0.186166,0.261279,0.552554,M102,SF,M104,M103
30,Senegal,Spain,0.653854,1.797308,2,1,Senegal,Spain,normal_time,0.129710,0.221430,0.648857,M104,Final,NaN,NaN


### Monte Carlo

In [24]:
#### Monte Carlo

def run_monte_carlo_no_conf_clean(n_simulations=100):
    wc_teams = sorted(df_groups["nation"].unique())

    stage_counts = {
        team: {
            "r32_count": 0,
            "r16_count": 0,
            "qf_count": 0,
            "sf_count": 0,
            "final_count": 0,
            "winner_count": 0,
        }
        for team in wc_teams
    }

    all_winners = []

    start_time = time.time()

    for i in range(n_simulations):
        tournament = simulate_tournament_no_conf_clean()
        summary = tournament["summary"]

        for team in summary["r32_teams"]:
            stage_counts[team]["r32_count"] += 1

        for team in summary["r16_teams"]:
            stage_counts[team]["r16_count"] += 1

        for team in summary["qf_teams"]:
            stage_counts[team]["qf_count"] += 1

        for team in summary["sf_teams"]:
            stage_counts[team]["sf_count"] += 1

        for team in summary["final_teams"]:
            stage_counts[team]["final_count"] += 1

        winner = summary["winner"]
        stage_counts[winner]["winner_count"] += 1
        all_winners.append(winner)

        if (i + 1) % 50 == 0:
            elapsed = time.time() - start_time
            print(f"Completed {i + 1}/{n_simulations} simulations in {elapsed:.1f}s")

    df_results = (
        pd.DataFrame.from_dict(stage_counts, orient="index")
        .reset_index()
        .rename(columns={"index": "team"})
    )

    probability_columns = {
        "r32_count": "r32_prob",
        "r16_count": "r16_prob",
        "qf_count": "qf_prob",
        "sf_count": "sf_prob",
        "final_count": "final_prob",
        "winner_count": "winner_prob",
    }

    for count_col, prob_col in probability_columns.items():
        df_results[prob_col] = df_results[count_col] / n_simulations

    df_results["confederation"] = df_results["team"].map(team_to_confederation)
    df_results["fifa_rank"] = df_results["team"].map(team_to_fifa_rank)
    df_results["rank_change"] = df_results["team"].map(team_to_fifa_rank_change)
    df_results["elo"] = df_results["team"].map(team_to_elo)
    df_results["form_score"] = df_results["team"].map(team_to_form)

    df_results = df_results.sort_values(
        "winner_prob",
        ascending=False,
    ).reset_index(drop=True)

    return df_results, all_winners

In [25]:
#### Run 100 simulations first

df_no_conf_clean_100, winners_no_conf_clean_100 = run_monte_carlo_no_conf_clean(
    n_simulations=100
)

display(
    df_no_conf_clean_100[
        [
            "team",
            "confederation",
            "fifa_rank",
            "elo",
            "r16_prob",
            "qf_prob",
            "sf_prob",
            "final_prob",
            "winner_prob",
        ]
    ].head(20)
)

df_no_conf_clean_100.groupby("confederation")["winner_prob"].sum().sort_values(ascending=False)

Completed 50/100 simulations in 29.4s
Completed 100/100 simulations in 59.6s


,team,confederation,fifa_rank,elo,r16_prob,qf_prob,sf_prob,final_prob,winner_prob
0,Spain,UEFA,2,2229.756779,0.78,0.55,0.48,0.39,0.31
1,France,UEFA,1,2124.778622,0.76,0.49,0.35,0.21,0.17
2,Argentina,CONMEBOL,3,2179.651451,0.67,0.61,0.49,0.30,0.17
3,Colombia,CONMEBOL,13,2064.025447,0.68,0.43,0.21,0.11,0.07
4,Brazil,CONMEBOL,6,2050.192145,0.65,0.44,0.27,0.09,0.05
5,England,UEFA,4,2094.723009,0.62,0.38,0.21,0.12,0.04
6,Netherlands,UEFA,7,2019.419663,0.54,0.30,0.21,0.08,0.03
7,South Korea,AFC,25,1877.973452,0.48,0.28,0.08,0.03,0.03
8,Japan,AFC,18,1963.035668,0.37,0.18,0.11,0.05,0.02
9,Ecuador,CONMEBOL,23,2015.364338,0.62,0.27,0.17,0.06,0.02


confederation
UEFA        0.59
CONMEBOL    0.31
AFC         0.05
CONCACAF    0.03
CAF         0.02
OFC         0.00
Name: winner_prob, dtype: float64

In [26]:
#### Code cell 20 — Sanity checks

print("Winner prob sum:", df_no_conf_clean_100["winner_prob"].sum())
print("Final prob sum:", df_no_conf_clean_100["final_prob"].sum())
print("SF prob sum:", df_no_conf_clean_100["sf_prob"].sum())
print("QF prob sum:", df_no_conf_clean_100["qf_prob"].sum())
print("R16 prob sum:", df_no_conf_clean_100["r16_prob"].sum())
print("R32 prob sum:", df_no_conf_clean_100["r32_prob"].sum())

Winner prob sum: 1.0
Final prob sum: 2.0
SF prob sum: 4.0
QF prob sum: 8.0
R16 prob sum: 16.0
R32 prob sum: 32.0


In [27]:
#### Run 1,000 simulations

df_no_conf_clean_1000, winners_no_conf_clean_1000 = run_monte_carlo_no_conf_clean(
    n_simulations=1000
)

display(
    df_no_conf_clean_1000[
        [
            "team",
            "confederation",
            "fifa_rank",
            "elo",
            "r32_prob",
            "r16_prob",
            "qf_prob",
            "sf_prob",
            "final_prob",
            "winner_prob",
        ]
    ].head(25)
)

Completed 50/1000 simulations in 30.1s
Completed 100/1000 simulations in 60.0s
Completed 150/1000 simulations in 89.8s
Completed 200/1000 simulations in 119.1s
Completed 250/1000 simulations in 149.0s
Completed 300/1000 simulations in 180.4s
Completed 350/1000 simulations in 211.1s
Completed 400/1000 simulations in 240.6s
Completed 450/1000 simulations in 270.2s
Completed 500/1000 simulations in 299.4s
Completed 550/1000 simulations in 330.0s
Completed 600/1000 simulations in 359.4s
Completed 650/1000 simulations in 388.5s
Completed 700/1000 simulations in 417.6s
Completed 750/1000 simulations in 446.6s
Completed 800/1000 simulations in 475.6s
Completed 850/1000 simulations in 504.6s
Completed 900/1000 simulations in 533.6s
Completed 950/1000 simulations in 562.5s
Completed 1000/1000 simulations in 591.6s


,team,confederation,fifa_rank,elo,r32_prob,r16_prob,qf_prob,sf_prob,final_prob,winner_prob
0,Spain,UEFA,2,2229.756779,0.999,0.808,0.612,0.509,0.386,0.269
1,Argentina,CONMEBOL,3,2179.651451,0.986,0.719,0.605,0.466,0.302,0.182
2,France,UEFA,1,2124.778622,0.951,0.724,0.475,0.332,0.193,0.116
3,England,UEFA,4,2094.723009,0.966,0.697,0.452,0.278,0.157,0.069
4,Brazil,CONMEBOL,6,2050.192145,0.955,0.644,0.416,0.238,0.123,0.049
5,Colombia,CONMEBOL,13,2064.025447,0.922,0.631,0.357,0.178,0.084,0.041
6,Netherlands,UEFA,7,2019.419663,0.919,0.556,0.351,0.185,0.078,0.038
7,Germany,UEFA,10,1987.742504,0.951,0.608,0.308,0.164,0.071,0.029
8,Ecuador,CONMEBOL,23,2015.364338,0.963,0.652,0.355,0.223,0.084,0.028
9,Portugal,UEFA,5,2023.470091,0.906,0.545,0.291,0.141,0.063,0.026


#### This is clearly better than baseline results because Confederation included bias for south american nations

In [29]:
#### Compare confederation winner probability

confed_winner_probs_no_conf_clean = (
    df_no_conf_clean_1000
    .groupby("confederation")["winner_prob"]
    .sum()
    .sort_values(ascending=False)
)

confed_stage_means_no_conf_clean = (
    df_no_conf_clean_1000
    .groupby("confederation")[
        ["r16_prob", "qf_prob", "sf_prob", "final_prob", "winner_prob"]
    ]
    .mean()
    .sort_values("winner_prob", ascending=False)
)

display(confed_winner_probs_no_conf_clean)
display(confed_stage_means_no_conf_clean)

confederation
UEFA        0.607
CONMEBOL    0.325
CAF         0.026
AFC         0.021
CONCACAF    0.020
OFC         0.001
Name: winner_prob, dtype: float64

,r16_prob,qf_prob,sf_prob,final_prob,winner_prob
confederation,,,,,
CONMEBOL,0.567333,0.350833,0.212833,0.112833,0.054167
UEFA,0.447562,0.242625,0.129750,0.069250,0.037937
CONCACAF,0.311167,0.122500,0.038167,0.012167,0.003333
CAF,0.162500,0.057900,0.021400,0.008300,0.002600
AFC,0.196556,0.073556,0.021889,0.006444,0.002333
OFC,0.174000,0.037000,0.007000,0.001000,0.001000


#### The key improvement is that winner probability is no longer only split between CONMEBOL and UEFA. Other confederations now have small but non-zero chances, which feels more realistic for a World Cup simulation.

### The big lesson: Removing confederation features fixed a large part of the regional bias.

In [30]:
#### Save A2 experiment output

OUTPUT_PATH = EXPERIMENT_OUTPUT_DIR / "no_conf_clean_probs.csv"

df_no_conf_clean_1000.to_csv(OUTPUT_PATH, index=False)

print("Saved no-conf clean tournament probabilities to:")
print(OUTPUT_PATH)

Saved no-conf clean tournament probabilities to:
/Users/riadanas/Desktop/Football WC2026/data/processed/experiments/no_conf_clean_probs.csv
